# Week 11 - AgentMesh API Walkthrough

**Applied GenAI & Agentic AI Engineering Course**

AgentMesh wraps the Week 10 TriageFlow specialist in an A2A-compatible service plus a cohort-shared MCP server. This notebook walks every endpoint with `curl` and Python `requests` side-by-side.

Start the server first:

```bash
uvicorn app.main:app --reload --reload-dir app --port 8000
```

> **Running against a tunnel or a classmate?** Every cell below reads `BASE` from the
> `AGENTMESH_BASE_URL` environment variable, falling back to `http://localhost:8000`.
> Set it before launching Jupyter and the whole notebook - curl cells included - follows.
> Point it at a peer's URL and you are driving *their* agent over A2A. See `SESSION.md`.

In [1]:
# Setup
import requests, json, os

# Same env var the server reads, so one export keeps the service and this notebook
# pointed at the same place. Leave it unset for a plain local run; set it to your
# cloudflared URL when tunnelling, or to a CLASSMATE's URL to drive their agent.
#   PowerShell:  $env:AGENTMESH_BASE_URL = "https://xxx.trycloudflare.com"
#   bash / zsh:  export AGENTMESH_BASE_URL=https://xxx.trycloudflare.com
BASE = os.environ.get("AGENTMESH_BASE_URL", "http://localhost:8000").rstrip("/")

TOKEN = os.environ.get("DEMO_TOKEN", "demo-token")  # in real deploys, JWT from your IdP
AUTH = {"Authorization": f"Bearer {TOKEN}"}
print("BASE:", BASE)

BASE: http://localhost:8000


## Section 1 - Health Check

Liveness + the pinned model id (shown in the UI health chip).

In [2]:
# curl
!curl -s {BASE}/health | python -m json.tool

{
    "status": "ok",
    "model": "gpt-5.4-mini-2026-03-17",
    "models": {
        "triage": "gpt-5.4-nano-2026-03-17",
        "knowledge": "gpt-5.4-mini-2026-03-17"
    },
    "auth_mode": "dev",
    "a2a_version": "1.0"
}


In [3]:
# requests
r = requests.get(f"{BASE}/health")
print(r.status_code)
print(json.dumps(r.json(), indent=2))

200
{
  "status": "ok",
  "model": "gpt-5.4-mini-2026-03-17",
  "models": {
    "triage": "gpt-5.4-nano-2026-03-17",
    "knowledge": "gpt-5.4-mini-2026-03-17"
  },
  "auth_mode": "dev",
  "a2a_version": "1.0"
}


## Section 2 - Agent Card (Discovery)

The A2A discovery surface. Unauthed - anyone on the network can fetch it.

In [4]:
!curl -s {BASE}/.well-known/agent-card.json | python -m json.tool

{
    "name": "agentmesh-stu_000",
    "version": "1.0.0",
    "protocolVersion": "1.0",
    "description": "AgentMesh instance operated by Unnamed Student (stu_000), specialty incident-triage. Triages incoming engineering incidents: classifies into knowledge, action, or escalate; retrieves runbooks for knowledge requests; proposes remediation actions behind a human approval gate; surfaces escalations directly.",
    "url": "http://localhost:8000",
    "provider": {
        "organization": "Unnamed Student",
        "url": "http://localhost:8000"
    },
    "skills": [
        {
            "id": "triage_incident",
            "name": "Triage incident",
            "description": "Triage one production incident. Use when an operator describes a production issue. Returns a category (knowledge / action / escalate), a summary, and either runbook citations or an approved remediation. Mutating actions pause at a human approval gate before anything runs.",
            "tags": [
             

In [5]:
card = requests.get(f"{BASE}/.well-known/agent-card.json").json()
print("name:", card["name"], "version:", card["version"], "protocol:", card["protocolVersion"])
# skills[] is what the agent DOES...
for skill in card["skills"]:
    print(" •", skill["id"], "-", skill["description"][:60], "…")
    print("   tags:", skill["tags"], "| inputModes:", skill["inputModes"])
# ...capabilities is protocol flags ONLY. No latency, no cost, no skill list.
print("capabilities:", card["capabilities"])
# supportedInterfaces[] is the v1.0 reachability field, and it is ORDERED:
# entry zero is the preferred one. A card with a single top-level "url" and no
# supportedInterfaces is a pre-v0.3.0 card.
for n, iface in enumerate(card["supportedInterfaces"]):
    tag = "preferred" if n == 0 else f"fallback {n}"
    print(f"  interface[{n}] {tag}:", iface["protocolBinding"], "v" + iface["protocolVersion"], iface["url"])
# Note the PLURAL. v1.0 carries signatures[], a list, because key rotation means
# a card may legitimately carry two valid signatures at once.
print("signatures:", card["signatures"], "(declared, unpopulated - this build does not sign its card)")


name: agentmesh-stu_000 version: 1.0.0 protocol: 1.0
 • triage_incident - Triage one production incident. Use when an operator describ …
   tags: ['incident', 'reliability', 'sre'] | inputModes: ['application/json', 'text/plain']
 • whoami - Return the identity of the student who runs this agent - stu …
   tags: ['identity', 'cohort', 'discovery'] | inputModes: ['application/json']
capabilities: {'streaming': True, 'pushNotifications': False, 'extendedAgentCard': False}
signature: None (declared, unpopulated - this build does not sign its card)


## Section 3 - Submit a Task (knowledge question, no approval)

`POST /tasks` validates the body against the skill's input schema and gates on JWT.
Returns `202 Accepted` with a task id and a stream URL, and the state `TASK_STATE_SUBMITTED`.

The body field is **`skill`** (A2A v1.0). `capability` is still accepted as a legacy alias.


In [6]:
body = {
  "skill": "triage_incident",
  "input": {
    "description": "Where is the runbook for restarting payments-api?",
    "severity": "low",
    "user_id": "u_42"
  }
}
r = requests.post(f"{BASE}/tasks", headers={**AUTH, "Content-Type": "application/json"}, json=body)
print(r.status_code, "| A2A-Version:", r.headers.get("A2A-Version"))
ack = r.json()
print(json.dumps(ack, indent=2))
task_id = ack.get("task_id")


202 | A2A-Version: 1.0
{
  "task_id": "task_13ae13bbb2a2",
  "state": "TASK_STATE_SUBMITTED",
  "stream_url": "http://localhost:8000/tasks/task_13ae13bbb2a2/stream"
}


## Section 4 - Stream the Task (SSE)

`GET /tasks/{task_id}/stream` emits state-transition events as they happen. Every event
carries an `id:` - a **per-task monotonic counter**, never the buffer's length - and the
server honours `Last-Event-ID` (header or `last_event_id` query param) for replay-on-reconnect.

State values are the A2A v1.0 wire strings: `TASK_STATE_*`. There is no lowercase form.


In [7]:
# Stream with requests (line-by-line iteration over the SSE response)
TERMINAL = {"TASK_STATE_COMPLETED", "TASK_STATE_FAILED", "TASK_STATE_CANCELED", "TASK_STATE_REJECTED"}

if task_id:
    with requests.get(f"{BASE}/tasks/{task_id}/stream", headers=AUTH, stream=True, timeout=30) as r:
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            if line.startswith("data:"):
                data = json.loads(line[5:].strip())
                print(f"#{data['event_id']:<3} {data['state']:26s} {data.get('payload', {}).get('message','')}")
                if data["state"] in TERMINAL:
                    break


#1   TASK_STATE_SUBMITTED       
#2   TASK_STATE_WORKING         classifying
#3   TASK_STATE_WORKING         retrieving
#4   TASK_STATE_COMPLETED       


## Section 5 - Action Request (hits the approval gate)

A mutating action description classifies as `action`, pauses at `TASK_STATE_INPUT_REQUIRED`,
and resumes only after `POST /tasks/{task_id}/input` arrives.

**Wait for the pause before replying.** Posting the approval the instant the 202 lands races
the background task and earns a 409 - the task is not paused *yet*.


In [8]:
body = {
  "skill": "triage_incident",
  "input": {
    "description": "Please restart payments-api now",
    "severity": "high",
    "user_id": "u_42"
  }
}
ack = requests.post(f"{BASE}/tasks", headers={**AUTH, "Content-Type": "application/json"}, json=body).json()
print(json.dumps(ack, indent=2))
action_task = ack["task_id"]


{
  "task_id": "task_88f04350783f",
  "state": "TASK_STATE_SUBMITTED",
  "stream_url": "http://localhost:8000/tasks/task_88f04350783f/stream"
}


In [9]:
# Wait for TASK_STATE_INPUT_REQUIRED on the stream, THEN approve.
proposal = None
with requests.get(f"{BASE}/tasks/{action_task}/stream", headers=AUTH, stream=True, timeout=30) as r:
    for line in r.iter_lines(decode_unicode=True):
        if line and line.startswith("data:"):
            data = json.loads(line[5:].strip())
            print(f"#{data['event_id']:<3} {data['state']}")
            if data["state"] == "TASK_STATE_INPUT_REQUIRED":
                proposal = data["payload"]["proposal"]
                break

print("\nproposal:", json.dumps(proposal, indent=2))

approval = {"approved": True, "note": "verified on-call"}
r = requests.post(f"{BASE}/tasks/{action_task}/input",
                  headers={**AUTH, "Content-Type": "application/json"},
                  json=approval)
print("approve ->", r.status_code, r.json())

# Reply `{"approved": false}` instead and the task lands in TASK_STATE_REJECTED -
# terminal, with nothing executed.


#1   TASK_STATE_SUBMITTED
#2   TASK_STATE_WORKING
#3   TASK_STATE_WORKING
#4   TASK_STATE_INPUT_REQUIRED

proposal: {
  "remediation": "restart",
  "target": "payments-api",
  "mutating": true,
  "requires_approval": true
}
approve -> 200 {'accepted': True}


## Section 6 - Failure Modes

Three failure shapes, all structured - `{"detail": {"reason": ..., "message": ...}}`:

1. **422** - wrong-shape body (severity as a freeform string). Field-level detail.
2. **401** - missing or invalid bearer token. `reason: missing_bearer`.
3. **409** - `POST /tasks/{task_id}/input` against a task that is not paused. `reason: task_not_paused`.


In [10]:
# Failure A - invalid severity (422, field-level)
bad = {"skill": "triage_incident", "input": {"description": "x", "severity": "urgent", "user_id": "u"}}
r = requests.post(f"{BASE}/tasks", headers={**AUTH, "Content-Type": "application/json"}, json=bad)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:400])


422
{
  "detail": [
    {
      "type": "string_too_short",
      "loc": [
        "description"
      ],
      "msg": "String should have at least 10 characters",
      "input": "x",
      "ctx": {
        "min_length": 10
      },
      "url": "https://errors.pydantic.dev/2.13/v/string_too_short"
    },
    {
      "type": "enum",
      "loc": [
        "severity"
      ],
      "msg": "Input should


In [11]:
# Failure B - missing bearer (401)
r = requests.post(f"{BASE}/tasks", headers={"Content-Type": "application/json"}, json=body)
print(r.status_code, r.json())


401 {'detail': {'reason': 'missing_bearer', 'message': 'Authorization: Bearer <token> required'}}


In [12]:
# Failure C - reply to a task that is not paused (409)
# `action_task` completed above, so it is no longer waiting on input.
r = requests.post(f"{BASE}/tasks/{action_task}/input",
                  headers={**AUTH, "Content-Type": "application/json"},
                  json={"approved": True})
print(r.status_code, json.dumps(r.json(), indent=2))


409 {
  "detail": {
    "reason": "task_not_paused",
    "current_state": "TASK_STATE_COMPLETED"
  }
}


## Section 7 - Swagger / OpenAPI

Run the cell below to get links that follow `BASE`, so they still work when you are
tunnelling rather than on localhost.

In [13]:
from IPython.display import display, HTML
display(HTML(
    f'<a href="{BASE}/docs" target="_blank" style="font-size:15px">Swagger UI: {BASE}/docs</a><br>'
    f'<a href="{BASE}/readme" target="_blank" style="font-size:15px">README: {BASE}/readme</a>'
))

---
## Section 8 - Who else is out there?

`cohort.json` holds every URL this project needs. Which mode you sweep in is set
by `COHORT_MODE` in `.env`:

- **solo** - localhost only, with local dummy peers. No tunnel, no classmates needed.
- **online** - your cloudflared URL plus the class's published `index.json`.

`cohort_roster.py` reads whichever mode is active, fetches each agent's Card, and runs
an authenticated `whoami` against it - all in parallel. It is the fastest way to answer
"who is up, and whose token is wrong?"

Full runbook in **`SESSION.md`**; the protocols themselves in **`A2A.md`** and **`MCP.md`**.


In [16]:
# Sweep whoever cohort.json points at (solo peers, or the whole class when online)
!python scripts/cohort_roster.py

# Just the cards, no auth and no task submitted:
#   !python scripts/cohort_roster.py --cards-only
#
# Ask one specific peer who they are:
#   !python scripts/a2a_client.py --peer stu_001 --skill whoami


solo mode: probing 2 peer(s) from cohort.json

id         name                 card   whoami   note
------------------------------------------------------------------------------
stu_001    Alice                ok     ok       
stu_002    Bob                  ok     ok       

2/2 card(s) reachable, 2 answered whoami


---
### Which mode am I in?

Everything above works on localhost. **Sections 9-13 are the live class**, and whether
they hit real classmates or local dummies is decided by `COHORT_MODE` in `.env`. Run
this to see where you stand before you go on.


In [ ]:
from app.cohort import active_mode, load_cohort

mode = active_mode()                      # reads COHORT_MODE from .env (solo | online)
print(f"COHORT_MODE = {mode}")
print()

try:
    c = load_cohort()
    print(f"  your base URL      : {c.my_base_url}")
    print(f"  coordination site  : {c.boss.site_url if c.boss else '(none configured)'}")
    print()
    if mode == "solo":
        print("SOLO. The cells below still run, but 'the cohort' is the local dummy")
        print("peers on 8001/8002. When the live session starts: set COHORT_MODE=online")
        print("in .env, point AGENTMESH_BASE_URL at your tunnel, and re-run from here.")
    else:
        print("ONLINE. Publishing to and reading from the real class roster.")
        print(f"Check AGENTMESH_BASE_URL matches {c.my_base_url} and your tunnel is up.")
except ValueError as e:
    # The classic online-mode slip: mode flipped, tunnel URL not filled in yet.
    print(f"COHORT_MODE={mode}, but the config is not ready:")
    print(f"  {e}")


---
## Section 9 - The cohort session (sign in, publish your URL)

These cells drive the same modules the command line uses, so there is one
implementation and no copy-paste. No coordination site in your class? Skip to
Section 11 and pass a peer URL directly.

> `asyncio.run()` does **not** work in Jupyter - a loop is already running. That is why
> the cells below use `await` at the top level, which Jupyter supports.


In [ ]:
# Sign in to the cohort site. The session is cached in .boss_session.json (gitignored).
from scripts.boss import Boss

boss = Boss()
print(boss.login("user@gmail.com", "pass"))
boss.whoami()


signed in as user@gmail.com


{'callsign': 'steady-shark-51',
 'name': 'steady-shark-51',
 'email': 'user@gmail.com',
 'uid': 'iRP4yxxxxxxxxxxxxqAXbpl1',
 'scope': 'triage:invoke mcp:invoke',
 'scope_ok': True,
 'note': ''}

In [3]:
# Publish your tunnel URL. Re-run this every time cloudflared restarts.
boss.publish("https://papua-portions-lovely-interaction.trycloudflare.com")


{'tag': 'steady-shark-51',
 'base_url': 'https://papua-portions-lovely-interaction.trycloudflare.com',
 'mcp': 'https://papua-portions-lovely-interaction.trycloudflare.com/mcp/',
 'student_name': 'steady-shark-51',
 'updated_at': 1786569529870}

In [4]:
# The exact block to paste into .env (identity + Firebase JWT settings)
print(boss.env_block())


STUDENT_ID=steady-shark-51
STUDENT_NAME=steady-shark-51
STUDENT_SPECIALTY=incident-triage

AUTH_MODE=jwks
JWT_ISSUER=https://securetoken.google.com/coresmart-agentmesh
JWT_AUDIENCE=coresmart-agentmesh
JWT_JWKS_URL=https://www.googleapis.com/service_accounts/v1/jwk/securetoken@system.gserviceaccount.com



---
## Section 10 - Who else is on the mesh

Fetches every agent's Card and runs an authenticated `whoami` against it. Run this
*before* everyone starts calling each other: it turns "it doesn't work" into a named
list of whose tunnel is down and whose token is wrong.


In [5]:
import httpx
from scripts.cohort_roster import _probe, _render, normalise_index
from app.cohort import load_cohort

c = load_cohort()
TOKEN = boss.token() if "boss" in dir() else "demo-token"

async with httpx.AsyncClient(follow_redirects=True) as client:
    if c.cohort_index_url:                       # online mode: the class roster
        r = await client.get(c.cohort_index_url, timeout=12.0)
        entries = normalise_index(r.json())
    else:                                        # solo mode: local dummy peers
        entries = [p.model_dump() for p in c.peers]
    rows = [await _probe(client, e, TOKEN, cards_only=False) for e in entries]

_render(rows, cards_only=False)


id         name                 card   whoami   note
------------------------------------------------------------------------------
stu_001    Alice (local dummy)  FAIL   -        unreachable: ConnectError
stu_002    Bob (local dummy)    FAIL   -        unreachable: ConnectError

0/2 card(s) reachable, 0 answered whoami


---
## Section 11 - Call a classmate's agent

The full A2A round trip: discover -> submit -> stream -> answer *their* approval gate.
Set `PEER` to a base URL from Section 10.

Watch the states: `SUBMITTED -> WORKING -> INPUT_REQUIRED`, then your approval crosses
the wire and their agent resumes to `COMPLETED`. Two machines, one task.


In [6]:
from scripts.a2a_client import (
    fetch_card, submit_task, follow_stream, resolve_stream_url, validate_card,
)

PEER  = "https://papua-portions-lovely-interaction.trycloudflare.com"   # <- a classmate's base URL
SKILL = "triage_incident"

async with httpx.AsyncClient(follow_redirects=True) as client:
    card = await fetch_card(client, PEER)
    print(f"{card['name']} - operated by {(card.get('provider') or {}).get('organization', '?')}")
    for problem in validate_card(card, SKILL, PEER):
        print("  WARNING:", problem)

    ack = await submit_task(client, PEER, TOKEN, SKILL, {
        "description": "Please restart payments-api now",
        "severity": "high",
        "user_id": "notebook",
    })
    task_id = ack["task_id"]
    stream_url, warning = resolve_stream_url(PEER, task_id, ack)
    if warning:
        print("  WARNING:", warning)

    # mode="approve" answers the gate automatically. Use "reject" to watch
    # TASK_STATE_REJECTED, with nothing executed on their side.
    final = await follow_stream(client, PEER, TOKEN, task_id, stream_url, "approve")

print("terminal state:", final)


agentmesh-steady-shark-51 - operated by steady-shark-51
  A2A-Version: 1.0
  #1   TASK_STATE_SUBMITTED       
  #2   TASK_STATE_WORKING         classifying
  #3   TASK_STATE_WORKING         proposing
  #4   TASK_STATE_INPUT_REQUIRED  
      proposal: {"remediation": "restart", "target": "payments-api", "mutating": true, "requires_approval": true}
  -> approved remotely
  ...resubscribing from event #4
  #5   TASK_STATE_WORKING         executing
  #6   TASK_STATE_COMPLETED       
      result: {"category": "action", "summary": "Executed: restart on payments-api", "proposed_action": "restart", "citations": [], "user_id": "notebook", "approved": true}
terminal state: TASK_STATE_COMPLETED


---
## Section 12 - Your own MCP surface (solo)

MCP is the *second* protocol this service speaks, co-hosted on the same port at `/mcp/`.
Everything so far drove the A2A task API; these cells drive the MCP tool / resource /
prompt surface - on **your own** server, so it works in solo mode with no classmates.

The three-call shape every MCP host uses is always the same: `initialize`, then `list`,
then `call`. Note the bearer - the co-hosted surface is gated (a public tunnel would
otherwise expose your tools to anyone), unlike the local `python -m app.mcp_server`
stdio server, which needs no auth because stdio never leaves your machine.


In [ ]:
# Jupyter cannot run an MCP client on its own event loop - the kernel's execution
# thread has too small a stack and the client's nesting overflows it (the kernel
# dies with 0xC00000FD and no traceback). The fix: give the client its own thread,
# its own event loop and a bigger stack. Defined here, reused by Section 13.
import asyncio, sys, threading

def run_mcp(factory, stack_mb=32):
    box = {}
    def worker():
        if sys.platform == "win32":
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        try:
            box["result"] = asyncio.run(factory())
        except BaseException as exc:
            box["error"] = exc
    threading.stack_size(stack_mb * 1024 * 1024)
    t = threading.Thread(target=worker)
    t.start(); t.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# One driver, used against your own server here and a classmate's in Section 13 - MCP
# is a protocol, so the only thing that changes between them is the URL. The trailing
# slash on /mcp/ is deliberate: the mount serves its route at "/", so ".../mcp" costs
# a 307 redirect on every call.
async def drive_mcp(url):
    async with streamablehttp_client(url, headers={"Authorization": f"Bearer {TOKEN}"}) as (r, w, _):
        async with ClientSession(r, w) as s:
            await s.initialize()
            tools = [t.name for t in (await s.list_tools()).tools]
            resources = [str(x.uri) for x in (await s.list_resources()).resources]
            prompts = [x.name for x in (await s.list_prompts()).prompts]
            # A tool is invoked; a resource is READ (no model in the loop); a prompt is
            # FETCHED as a ready-made template. One of each, so all three primitives show.
            whoami = (await s.call_tool("whoami", {})).structuredContent
            incident = (await s.read_resource("agentmesh://incident/INC-102")).contents[0].text
            brief = (await s.get_prompt(
                "triage_brief", {"description": "primary db replication lag", "severity": "high"}
            )).messages[0].content.text
            return {"tools": tools, "resources": resources, "prompts": prompts,
                    "whoami": whoami, "incident": incident, "brief": brief}

me = run_mcp(lambda: drive_mcp(f"{BASE}/mcp/"))
print("tools    :", me["tools"])
print("resources:", me["resources"])
print("prompts  :", me["prompts"])
print("whoami   :", me["whoami"])
print("read agentmesh://incident/INC-102 ->", me["incident"])
print("prompt triage_brief ->", me["brief"][:80], "...")


---
## Section 13 - A classmate's MCP surface (online)

The **identical** client, pointed at a classmate's tunnel instead of your own. That is
the whole point of a protocol: the same `initialize -> list -> call` drives their
server over the internet exactly as it drove yours. Set `PEER` in Section 11 first.

The bearer matters more here: their co-hosted `/mcp/` is on the public internet, so it
verifies your token and checks the `mcp:invoke` scope before running a thing.


In [ ]:
try:
    PEER
except NameError:
    print("PEER is not set. This is the ONLINE twin of Section 12 - run Section 11")
    print("first to pick a classmate, or stay in solo: Section 12 already drove the")
    print("identical client against your own server.")
else:
    them = run_mcp(lambda: drive_mcp(f"{PEER}/mcp/"))
    print("their tools    :", them["tools"])
    print("their resources:", them["resources"])
    print("their prompts  :", them["prompts"])
    print("their whoami   :", them["whoami"])
